### `test_rk_solvers.ipynb` 
*Created: Sept 24, 2026* <br/>
This notebooks implements convergence and accuracy tests for some custom ODE solvers by comparing them against exact solutions when available, and against algorithms in `OrdinaryDiffEq.jl` when an exact solution is not available.

In [7]:
@time using OrdinaryDiffEq
@time import OrdinaryDiffEqCore: OrdinaryDiffEqAlgorithm  
@time using CairoMakie, NBInclude, UnPack, Printf, Test, LinearAlgebra, LaTeXStrings, Statistics

  0.001517 seconds (326 allocations: 18.234 KiB)
  0.001476 seconds (326 allocations: 18.242 KiB)
  0.009970 seconds (2.96 k allocations: 171.664 KiB)


In [6]:
using OrdinaryDiffEqExponentialRK

# 1. Define the stiff linear operator A (Matrix or SciMLOperator)
A = MatrixOperator([-100.0 1.0; 
       0.0 -0.01])

# 2. Define the non-stiff nonlinear function g(u, p, t)
function g(u, p, t)
    return [sin(u[2]), cos(u[1])]
end

u0 = [1.0, 1.0]
tspan = (0.0, 10.0)

# 3. Create the SplitODEProblem (Linear part first, nonlinear second)
prob = SplitODEProblem(A, g, u0, tspan)

# 4. Solve using LawsonEuler (Note: Exponential algorithms use fixed timestepping)
dt = 0.01
sol = solve(prob, NorsettEuler(), dt=dt);


In [8]:
#Import solvers
@nbinclude("euler.ipynb")
@nbinclude("rk4.ipynb")

#Import ODE test problems
@nbinclude("test_problems.ipynb");
@nbinclude("../../../tests.ipynb")

In [9]:
function compare_solutions(prob::ODETestProblem, reference_alg::OrdinaryDiffEqAlgorithm, custom_alg::A; dt::Real = 0.01) where {A}
    """    
    reference_alg :: algorithm from the OrdinaryDiffEq package 
    custom_alg :: algorithm that I wrote, that I'm testing 
    """
  
    @unpack f, u0, tspan, p = prob

   
    custom_sol = custom_alg(f, u0, tspan, p; dt = dt)
    ref_sol = solve(ODEProblem(f, u0, tspan, p), reference_alg; adaptive = false, dt = dt, saveat = custom_sol.t)

    #STEP 3: Compute difference between reference solution and custom solution 
    u_custom = custom_sol.u
    u_ref = ref_sol.u
    max_l2_error = maximum(norm.(u_ref .- u_custom))

    return (reference_sol = ref_sol, custom_sol = custom_sol, max_l2_error = max_l2_error)     
end 

compare_solutions (generic function with 1 method)

In [13]:
results = run_tests(; title = "Runge-Kutta Tests") do suite

    reference_alg = Euler()
    custom_alg = euler

    # reference_alg = RK4()
    # custom_alg = rk4
    
    tol = 1e-8
    dt = 0.01

    test!(suite, "Exponential") do
        @test compare_solutions(exp_growth, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end

    test!(suite, "Sinusoid") do
        @test compare_solutions(sinusoid, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end

    test!(suite, "Damped Oscillator") do
        @test compare_solutions(damped_oscillator, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end

    test!(suite, "Lotka-Volterra") do
        @test compare_solutions(lotka_volterra, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end

    test!(suite, "Lorenz System") do
        @test compare_solutions(lorenz_system, reference_alg, custom_alg; dt = dt).max_l2_error ≤ tol
    end
end;

Runge-Kutta Tests
Test                Result      Time (s)
Exponential         PASS        4.45e-04
Sinusoid            PASS        4.37e-04
Damped Oscillator   PASS        1.57e-03
Lotka-Volterra      PASS        1.36e-03
Lorenz System       PASS        1.77e-03
────────────────────────────────────────
Total                           5.58e-03

Assertions/results: 5 passed · 0 failed · 0 errors
All tests passed.
